# 15 Keep the Best Weights Seen

| | |
|---|---|
| **Path** | Perceptron |
| **Prerequisite** | 14 |

Add a fourth paper, D, with scores $(4, 4)$ and label fail. The line of Lesson 07,

$$
s = 2x_1 + x_2 - 1
$$

scores D as

$$
s(D) = 8 + 4 - 1 = 11 > 0
$$

The line says pass. D failed. One paper is wrong, and the other three are right.

No line gets all four right: D sits deep on the pass side of every line that keeps A and B passing and C failing. The honest target is one mistake, not zero.

If the walk updates on D anyway,

$$
(2, 1, -1) + (-1)(4, 4, 1) = (-2, -3, -2)
$$

The new margins are

$$
\begin{aligned}
m(A) &= -9 \\
m(B) &= -10 \\
m(C) &= 2 \\
m(D) &= 22
\end{aligned}
$$

A and B are now wrong. The mistake count went from $1$ to $2$.

The **pocket** rule is a second memory. After every update, if the mistake count is the best count seen so far, keep those weights. Otherwise keep the previous champion. Here the champion stays $(2, 1, -1)$, with one mistake, and the current weights are not promoted.


## Example 1 — The damaging update

### Predict

The cell prints the old mistake count `1`, the new weights, and the new mistake count `2`.


In [1]:
# Hand: D pulls (2, 1, -1) to (-2, -3, -2), and the mistake count rises from 1 to 2.
papers = [((2, 1), 1, "A"), ((1, 2), 1, "B"), ((0, 0), -1, "C"), ((4, 4), -1, "D")]

def margins(weights):
    found = []
    for (x1, x2), label, name in papers:
        found.append((name, label * (weights[0] * x1 + weights[1] * x2 + weights[2])))
    return found

before = (2, 1, -1)
before_mistakes = sum(margin <= 0 for _, margin in margins(before))
after = tuple(before[i] + (-1) * phi for i, phi in enumerate((4, 4, 1)))
after_rows = margins(after)
after_mistakes = sum(margin <= 0 for _, margin in after_rows)
print(before_mistakes)
print(after, after_rows, after_mistakes)
assert before_mistakes == 1
assert after == (-2, -3, -2)
assert after_rows == [("A", -9), ("B", -10), ("C", 2), ("D", 22)]
assert after_mistakes == 2


1
(-2, -3, -2) [('A', -9), ('B', -10), ('C', 2), ('D', 22)] 2


## Example 2 — The pocket refuses the promotion

### Predict

The stored champion is still `(2, 1, -1)`.


In [2]:
# A pocket updates only when the new mistake count is strictly better.
champion = (2, 1, -1)
best = 1
candidate = (-2, -3, -2)
candidate_mistakes = 2
if candidate_mistakes < best:
    champion = candidate
    best = candidate_mistakes
print(champion, best)
assert champion == (2, 1, -1)
assert best == 1


(2, 1, -1) 1


## Pitfall — The pocket only remembers weights the walk has actually stood on

It does not search the plane for a better line. If the walk had never visited $(2, 1, -1)$, the pocket could not invent it. On the four-gate cycle of Lesson 13, the best count the cycle stands on is $2$ mistakes out of $4$. A different line, $(2, 2, -1)$, misses only the gate 11 and therefore has $1$ mistake, but that cycle never visits it. The pocket cannot keep a vector it has not seen.


In [3]:
# (2, 2, -1) misses only gate 11. The zero cycle never stood on this vector.
gates = [((0, 0), -1), ((1, 0), 1), ((0, 1), 1), ((1, 1), -1)]
weights = (2, 2, -1)
mistakes = 0
for (x1, x2), label in gates:
    margin = label * (weights[0] * x1 + weights[1] * x2 + weights[2])
    mistakes += margin <= 0
print(mistakes)
assert mistakes == 1


1


## Summary

One update on the far fail paper D turns a one-mistake line into a two-mistake line. The pocket keeps $(2, 1, -1)$. It is a memory of the best count seen on this walk, not a search for the best line in the plane.
